In [7]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

from imblearn.over_sampling import SMOTE


# =========================
# LOAD DATA
# =========================

data = pd.read_csv("data/customer_churn.csv")

data = data.dropna(subset=["Churn?"])


# =========================
# FEATURE ENGINEERING
# =========================

data["TotalMinutes"] = (
    data["Day Mins"] +
    data["Eve Mins"] +
    data["Night Mins"]
)

data["TotalCalls"] = (
    data["Day Calls"] +
    data["Eve Calls"] +
    data["Night Calls"]
)

data["HighServiceCalls"] = (
    data["CustServ Calls"] > 4
).astype(int)

data["AvgCallDuration"] = (
    data["TotalMinutes"] /
    data["TotalCalls"]
)


# =========================
# ENCODING
# =========================

le = LabelEncoder()

for col in data.select_dtypes(include="object").columns:
    if col != "Churn?":
        data[col] = le.fit_transform(data[col])

data["Churn?"] = data["Churn?"].map({
    "True.": 1,
    "False.": 0
})


# =========================
# FEATURES & TARGET
# =========================

customer_phone = data["Phone"]

X = data.drop(
    ["Churn?", "Phone"],
    axis=1
)

y = data["Churn?"]


# =========================
# TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test, phone_train, phone_test = train_test_split(
    X,
    y,
    customer_phone,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =========================
# SMOTE ONLY ON TRAIN DATA
# =========================

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)


# =========================
# MODEL TRAINING
# =========================

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(
    X_train_smote,
    y_train_smote
)

y_pred = rf_model.predict(X_test)

y_prob = rf_model.predict_proba(X_test)[:, 1] * 100

# =========================
# CUSTOMER RISK TABLE
# =========================

results = pd.DataFrame({
    "Phone": phone_test.values,
    "Actual": y_test.values,
    "Predicted": y_pred,
    "Churn_Probability(%)": y_prob
})


def risk_level(prob):
    if prob > 80:
        return "High Risk"
    elif prob >= 50:
        return "Medium Risk"
    else:
        return "Low Risk"


results["Risk_Level"] = results[
    "Churn_Probability(%)"
].apply(risk_level)

results = results.sort_values(
    by="Churn_Probability(%)",
    ascending=False
)

results.to_csv(
    "data/customer_risk.csv",
    index=False
)


# =========================
# FEATURE IMPORTANCE
# =========================

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

feature_importance.to_csv(
    "data/feature_importance.csv",
    index=False
)


# SAVE MODEL PERFORMANCE
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
performance = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
    ],
    "Value": [
        accuracy,
        precision,
        recall,
        f1
    ]
})
performance.to_csv(
    "data/performance.csv",
    index=False
)


# =========================
# CONFUSION MATRIX
# =========================

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='turbo'
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.savefig(
    "data/confusion_matrix.png",
    bbox_inches="tight"
)
plt.close()


# =========================
# FEATURE IMPORTANCE GRAPH
# =========================

plt.figure(figsize=(8, 6))

sns.barplot(
    data=feature_importance,
    x="Importance",
    y="Feature"
)

plt.title("Feature Importance")
plt.savefig(
    "data/feature_importance.png",
    bbox_inches="tight"
)
plt.close()